# Reference Wavefront: FAM Make Table

**Author:** Aaron Roodman  
**Date Created:** 2026-02-23  
**Last Modified:** 2026-03-15  
**Status:** [In Progress]  
**Keywords:** [AOS, Intrinsic Wavefront, Full Array Mode, Zernikes, Table]

## Description

This notebook creates a comprehensive table of Zernike wavefront measurements from Rubin Full Array Mode (FAM) cwfs images with model intrinsic values.

**Key functionality:**
1. Generate model intrinsic wavefront maps across the focal plane for each Zernike term
2. Identify FAM observations by test block program (e.g., T278, T381, T492, T539, T614)
3. Extract Zernike measurements from Butler for identified observations
4. Calculate per-visit mean Zernikes in configurable coordinate system (OCS or CCS)
5. Interpolate model intrinsic values at each donut position using field angles
6. Calculate residuals (data - model) for comparison
7. Determine rotator angles from ConsDB `physical_rotator_angle` and calculated fallback
8. Save comprehensive table to parquet file for analysis

**Output:** Parquet file with all Zernike data, ready for analysis in companion notebook

**Based on:**
- `reference_wavefront_FAM_study.ipynb`: Zernike wavefront extraction and intrinsic model
- `locate_test_blocks.ipynb`: Isolating Full Array Mode data by day_obs and seq_num
- LSSTCam AOS Datasets: https://rubinobs.atlassian.net/wiki/spaces/LTS/pages/761397307/LSSTCam+AOS+Datasets

<a id='changelog'></a>
## Change Log

| Date | Author | Description |
|------|--------|-------------|
| 2026-02-23 | Aaron Roodman | Initial version |
| 2026-03-13 | Aaron Roodman | Added parameter sets, rotator check |
| 2026-03-15 | Aaron Roodman | Added OCS/CCS parameter, revised rotator angle using ConsDB physical_rotator_angle and calculated fallback, output to output/ subdirectory |

## Table of Contents

- [Change Log](#changelog)
- [Parameters](#params)
- [Setup & Imports](#setup)
- [Helper Functions](#functions)
- [Model Intrinsic Wavefront](#intrinsic)
- [Identify FAM Data](#identify)
- [Rotator Angles](#rotator)
- [Extract Zernikes](#extract)
- [Add Model and Rotator Angles to Data](#model)
- [Results Table](#results)

<a id='params'></a>
## Parameters

All configurable parameters are collected here. Select `PARAM_SET` to choose which dataset to process.

In [ ]:
# ============================================================
# Parameters
# ============================================================

# Select parameter set (see intrinsics_lib.PARAM_SETS for all options)
PARAM_SET = 6
COORD_SYS = 'OCS'

# Optional overrides
INCLUDE_THERMAL = True
OUTPUT_DIR = 'output'

<a id='setup'></a>
## Setup & Imports

In [ ]:
# Setup
import sys
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy.stats import binned_statistic_2d
from pathlib import Path

sys.path.insert(0, 'code')
from intrinsics_lib import (
    run_mktable, PARAM_SETS, infer_zernike_indices, derive_version_strings,
)

params = PARAM_SETS[PARAM_SET]
print(f"Parameter set {PARAM_SET}: {params['prefix']} "
      f"{params['day_obs_min']}-{params['day_obs_max']}")
print(f"  Butler: {params['butler_repo']}")
print(f"  Collections: {params['fam_collections']}")

<a id='run'></a>
## Run Pipeline

In [ ]:
aosTable, visit_info = await run_mktable(
    butler_repo=params['butler_repo'],
    fam_collections=params['fam_collections'],
    day_obs_min=params['day_obs_min'],
    day_obs_max=params['day_obs_max'],
    fam_programs=params['fam_programs'],
    prefix=params['prefix'],
    coord_sys=COORD_SYS,
    output_dir=OUTPUT_DIR,
    include_thermal=INCLUDE_THERMAL,
)

<a id='results'></a>
## Results Table and Data vs Model Comparison

In [55]:
if aosTable is not None:
    print("Table columns:")
    print(sorted(aosTable.columns))
    print(f"\nTable shape: {len(aosTable)} rows")
    print("\nFirst 5 rows:")
    print(aosTable[:5])

Table columns:
['centroid_x', 'centroid_x_extra', 'centroid_x_intra', 'centroid_y', 'centroid_y_extra', 'centroid_y_intra', 'coord_dec', 'coord_ra', 'day_obs', 'detector', 'extra_fpx', 'extra_fpy', 'intra_fpx', 'intra_fpy', 'matched_intra_extra', 'rotator_angle', 'rotator_flagged', 'seq_num', 'snr', 'snr_extra', 'snr_intra', 'thx_CCS', 'thx_CCS_extra', 'thx_CCS_intra', 'thx_OCS', 'thx_OCS_extra', 'thx_OCS_intra', 'thy_CCS', 'thy_CCS_extra', 'thy_CCS_intra', 'thy_OCS', 'thy_OCS_extra', 'thy_OCS_intra', 'used', 'zk_CCS', 'zk_OCS', 'zk_OCS_mean', 'zk_deviation_CCS', 'zk_deviation_OCS', 'zk_intrinsic', 'zk_intrinsic_CCS', 'zk_intrinsic_OCS', 'zk_residual']

Table shape: 432093 rows

First 5 rows:
           zk_CCS                  zk_intrinsic_CCS        ... rotator_flagged
--------------------------- ------------------------------ ... ---------------
-0.30036104 .. -0.052650463  -0.06313663 .. -9.7856595e-05 ...           False
-0.31006485 .. -0.042412415 -0.060099717 .. -0.00012699398 ..

In [56]:
if aosTable is not None:
    print("\nMeasurements per day_obs:")
    day_counts = pd.DataFrame(aosTable['day_obs']).value_counts().sort_index()
    print(day_counts)


Measurements per day_obs:
day_obs 
20260315    178564
20260316     83422
20260317    170107
Name: count, dtype: int64


In [ ]:
if aosTable is not None:
    zk_col = f'zk_{COORD_SYS}'
    zk_data = np.stack(aosTable[zk_col])
    nZk_data = zk_data.shape[1]
    iZs = infer_zernike_indices(nZk_data)
    iZidx = {iZ: i for i, iZ in enumerate(iZs)}

    # Scalar columns
    scalar_cols = [col for col in aosTable.columns
                   if not hasattr(aosTable[col][0], '__len__') or isinstance(aosTable[col][0], str)]
    df_display = aosTable[scalar_cols].to_pandas()

    pd.options.display.float_format = '{:.2f}'.format
    pd.options.display.max_columns = None
    pd.options.display.width = None

    print(f"\nTable Preview (first 10 rows):")
    print(f"Total rows: {len(aosTable)}, Total columns: {len(aosTable.columns)}")
    print(f"\nScalar columns:")
    print(df_display.head(10))

    # Show first 8 Zernike terms
    n_show = min(8, nZk_data)
    zk_display = pd.DataFrame(
        zk_data[:10, :n_show],
        columns=[f'Z{iZ}' for iZ in iZs[:n_show]]
    )
    zk_display.insert(0, 'day_obs', aosTable['day_obs'][:10])
    zk_display.insert(1, 'seq_num', aosTable['seq_num'][:10])
    print(f"\nZernike Coefficients ({zk_col}, first {n_show} terms):")
    print(zk_display)
    print(f"\nFull Zernike indices ({nZk_data} terms): {iZs}")

    pd.reset_option('display.float_format')

In [ ]:
zk_col = f'zk_{COORD_SYS}'
# Z4 comparison: data vs model
if aosTable is not None:
    zk_data = np.stack(aosTable[zk_col])
    nZk_data = zk_data.shape[1]
    iZs = infer_zernike_indices(nZk_data)
    iZidx = {iZ: i for i, iZ in enumerate(iZs)}

    if 4 not in iZidx:
        print("Z4 not found in data Zernike indices!")
    else:
        z4_col = iZidx[4]
        Z4_data = zk_data[:, z4_col]
        Z4_model = np.array([row[z4_col] for row in aosTable['zk_intrinsic']]) * 1e6
        Z4_residual = Z4_data - Z4_model

        fig, axes = plt.subplots(1, 3, figsize=(15, 4))

        axes[0].hist(Z4_data, bins=50, edgecolor='black', alpha=0.7)
        axes[0].set_xlabel('Z4 (Defocus) [um]'); axes[0].set_ylabel('Count')
        axes[0].set_title('Data'); axes[0].grid(alpha=0.3)

        axes[1].hist(Z4_model, bins=50, edgecolor='black', alpha=0.7, color='orange')
        axes[1].set_xlabel('Z4 (Defocus) [um]'); axes[1].set_ylabel('Count')
        axes[1].set_title('Model Intrinsic'); axes[1].grid(alpha=0.3)

        axes[2].hist(Z4_residual, bins=50, edgecolor='black', alpha=0.7, color='green')
        axes[2].set_xlabel('Z4 Residual [um]'); axes[2].set_ylabel('Count')
        axes[2].set_title('Data - Model'); axes[2].grid(alpha=0.3)

        plt.suptitle(f'Z4 Comparison ({COORD_SYS}): FAM Data ({day_obs_min}-{day_obs_max})', y=1.02)
        plt.tight_layout()
        plt.show()

        print(f"\nZ4 statistics (nm) [{COORD_SYS}]:")
        print(f"{'':12} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
        for label, arr in [('Data', Z4_data), ('Model', Z4_model), ('Residual', Z4_residual)]:
            print(f"{label:12} {np.mean(arr)*1e6:10.1f} {np.std(arr)*1e6:10.1f} "
                  f"{np.min(arr)*1e6:10.1f} {np.max(arr)*1e6:10.1f}")